# CS171 Project — Model Construction  
## E-Waste Image Classification (Model V3 & Model V4)

**Course:** CS 171 — Introduction to Machine Learning  
**Name:** Samriddhi Matharu  
**Student ID:** 016328156  

---

## 1. Notebook Purpose

This notebook focuses on the **construction, training, and basic evaluation** of two models for my e-waste image classification project:

- **Model V3:** Custom Convolutional Neural Network (CNN)  
- **Model V4:** Pretrained ResNet-18 (transfer learning)

All **data preparation** (downloading the Kaggle E-Waste dataset, merging the Kaggle validation into the test set, and adding the hand-curated validation set from the web) was completed in a separate notebook:

- `01_data_preparation_ewaste.ipynb`

After running that notebook, the prepared data folders are:

- `data/train/` — Kaggle training images (10 e-waste classes)  
- `data/test/` — Kaggle test images + merged Kaggle validation images  
- `data/val (by hand)/` — Hand-curated real-world images collected from the web  

In this notebook, I will:

1. Load these prepared splits.  
2. Define Model V3 (custom CNN) and Model V4 (ResNet-18).  
3. On `data/train/`I will show the **training setup and original training output** for each model.
4. Load the previously saved trained weights.  
5. Evaluate each model on:
   - Kaggle **test** set (`data/test/`)  
   - Hand-curated **validation** set (`data/val (by hand)/`)
   - Accuracy  
   - Macro-averaged precision, recall, and F1 score  
   - Per-class classification report  

Plots and deeper analysis of the results (per-class breakdown, curves, confusion matrices, etc.) are handled in a **separate Analysis & Visualization notebook**.


## 1. Import Libraries

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cpu


In [2]:
# Paths prepared in 01_data_preparation_ewaste.ipynb

train_dir = "data/train"
test_dir  = "data/test"

print("Train directory:", train_dir)
print("Test directory :", test_dir)


Train directory: data/train
Test directory : data/test


## 2. Model V3 — Custom CNN

In [3]:
#  Transforms, Datasets, Loaders for Model V3 (128x128)
# ------------------------------

IMG_SIZE_V3 = 128
BATCH_SIZE_V3 = 32

train_transform_v3 = transforms.Compose([
    transforms.Resize((IMG_SIZE_V3, IMG_SIZE_V3)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

test_transform_v3 = transforms.Compose([
    transforms.Resize((IMG_SIZE_V3, IMG_SIZE_V3)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

train_ds_v3 = datasets.ImageFolder(root=train_dir, transform=train_transform_v3)
test_ds_v3  = datasets.ImageFolder(root=test_dir,  transform=test_transform_v3)

class_names_v3 = train_ds_v3.classes
num_classes_v3 = len(class_names_v3)

print("Classes (V3):", class_names_v3)
print(f"Train images: {len(train_ds_v3)}, Test images: {len(test_ds_v3)}")

train_loader_v3 = DataLoader(train_ds_v3, batch_size=BATCH_SIZE_V3, shuffle=True)
test_loader_v3  = DataLoader(test_ds_v3,  batch_size=BATCH_SIZE_V3, shuffle=False)


Classes (V3): ['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']
Train images: 2400, Test images: 600


### 2.1 Model V3 — Custom CNN Architecture

Model V3 is a custom Convolutional Neural Network designed for **128×128 RGB images**.

**Architecture summary:**

- **Feature extractor**
  - Block 1:
    - Conv2d(3 → 16), BatchNorm, ReLU  
    - Conv2d(16 → 32), BatchNorm, ReLU  
    - MaxPool2d(2) → spatial size: 128 → 64  
  - Block 2:
    - Conv2d(32 → 64), ReLU  
    - Conv2d(64 → 128), ReLU  
    - MaxPool2d(2) → spatial size: 64 → 32  

- **Classifier**
  - Flatten  
  - Linear → 256, ReLU, Dropout  
  - Linear → `num_classes_v3` (10 e-waste categories)

This model is trained **from scratch** on the Kaggle e-waste training set.


In [4]:
# ------------------------------
# Define the CNN Model (V3)
# ------------------------------

class EWasteCNNv3(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Feature extractor
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),      # 128 -> 64

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),      # 64 -> 32
        )

        # After 2 pooling layers: 128 channels of 32×32
        conv_out_size = 128 * 32 * 32

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_out_size, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate model, loss, optimizer
model_v3 = EWasteCNNv3(num_classes=num_classes_v3).to(device)
criterion_v3 = nn.CrossEntropyLoss()
optimizer_v3 = optim.Adam(model_v3.parameters(), lr=8e-4)

print("Model V3 Created!")
print(model_v3)


Model V3 Created!
EWasteCNNv3(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=131072, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.15, inplace=False)
    (4): Linear(in_features=256, out

In [5]:
# ------------------------------------
#  Training Loop for Model V3
# ------------------------------------

def compute_correct_labels(outputs, labels):
    """Returns (# of correct predictions, batch size)."""
    _, preds = torch.max(outputs, 1)
    return (preds == labels).sum().item(), labels.size(0)


def training_loop_v3(model, optimizer, num_epochs, train_loader, test_loader, printing=True):
    train_losses, test_losses = [], []
    train_accs,  test_accs  = [], []

    for epoch in range(num_epochs):
        # TRAINING PHASE
        model.train()
        total_train_loss = 0.0
        total_train_correct = 0
        total_train_images = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion_v3(outputs, yb)
            loss.backward()
            optimizer.step()

            correct, total = compute_correct_labels(outputs, yb)
            total_train_correct += correct
            total_train_images  += total
            total_train_loss    += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = total_train_correct / total_train_images
        train_losses.append(avg_train_loss)
        train_accs.append(train_acc)

        # TESTING PHASE 
        model.eval()
        total_test_loss = 0.0
        total_test_correct = 0
        total_test_images = 0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                loss = criterion_v3(outputs, yb)

                total_test_loss += loss.item()
                correct, total = compute_correct_labels(outputs, yb)
                total_test_correct += correct
                total_test_images  += total

        avg_test_loss = total_test_loss / len(test_loader)
        test_acc = total_test_correct / total_test_images
        test_losses.append(avg_test_loss)
        test_accs.append(test_acc)

        if printing:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Train Acc: {train_acc*100:.2f}% | "
                f"Test Loss: {avg_test_loss:.4f} | "
                f"Test Acc: {test_acc*100:.2f}%"
            )

    return train_losses, test_losses, train_accs, test_accs


In [ ]:
# OPTIONAL: How V3 was originally trained.
# This cell can stay commented out for grading to avoid long retraining.
# If you would like to re train, please uncomment the code below which runs the model and saves it 
# I already ran this model before and saved the model

# NUM_EPOCHS_V3 = 30
# train_losses_v3, test_losses_v3, train_accs_v3, test_accs_v3 = training_loop_v3(
#     model_v3,
#     optimizer_v3,
#     NUM_EPOCHS_V3,
#     train_loader_v3,
#     test_loader_v3,
#     printing=True
# )
#
# os.makedirs("models", exist_ok=True)
# save_path_v3 = "models/ewaste_cnn_v3.pth"
# torch.save(model_v3.state_dict(), save_path_v3)
# print(f"Model V3 saved to: {save_path_v3}")


### 2.2 Original Training Output for Model V3 (30 Epochs)

Below is the training and test performance from the original 30-epoch run of Model V3:

```text

Epoch 1/30 | Train Loss: 2.3437 | Train Acc: 24.83% | Test Loss: 1.7785 | Test Acc: 37.67%
Epoch 2/30 | Train Loss: 1.6763 | Train Acc: 42.71% | Test Loss: 1.8216 | Test Acc: 35.33%
Epoch 3/30 | Train Loss: 1.5195 | Train Acc: 46.29% | Test Loss: 1.4504 | Test Acc: 51.33%
Epoch 4/30 | Train Loss: 1.3695 | Train Acc: 52.29% | Test Loss: 1.4484 | Test Acc: 50.00%
Epoch 5/30 | Train Loss: 1.2367 | Train Acc: 57.00% | Test Loss: 1.3664 | Test Acc: 54.50%
Epoch 6/30 | Train Loss: 1.0667 | Train Acc: 64.62% | Test Loss: 1.1293 | Test Acc: 62.83%
Epoch 7/30 | Train Loss: 0.8708 | Train Acc: 70.04% | Test Loss: 1.1916 | Test Acc: 60.00%
Epoch 8/30 | Train Loss: 0.7769 | Train Acc: 73.25% | Test Loss: 1.1897 | Test Acc: 60.67%
Epoch 9/30 | Train Loss: 0.6528 | Train Acc: 78.12% | Test Loss: 1.0929 | Test Acc: 63.33%
Epoch 10/30 | Train Loss: 0.5549 | Train Acc: 81.71% | Test Loss: 1.2446 | Test Acc: 62.17%
Epoch 11/30 | Train Loss: 0.5410 | Train Acc: 81.08% | Test Loss: 1.2024 | Test Acc: 67.33%
Epoch 12/30 | Train Loss: 0.4738 | Train Acc: 84.42% | Test Loss: 1.2458 | Test Acc: 63.17%
Epoch 13/30 | Train Loss: 0.3937 | Train Acc: 87.38% | Test Loss: 1.2448 | Test Acc: 65.17%
Epoch 14/30 | Train Loss: 0.3067 | Train Acc: 89.25% | Test Loss: 1.3807 | Test Acc: 65.50%
Epoch 15/30 | Train Loss: 0.3062 | Train Acc: 90.04% | Test Loss: 1.4982 | Test Acc: 65.17%
Epoch 16/30 | Train Loss: 0.2790 | Train Acc: 90.54% | Test Loss: 1.3317 | Test Acc: 63.17%
Epoch 17/30 | Train Loss: 0.2406 | Train Acc: 91.50% | Test Loss: 1.3597 | Test Acc: 65.00%
Epoch 18/30 | Train Loss: 0.2532 | Train Acc: 91.25% | Test Loss: 1.5883 | Test Acc: 63.50%
Epoch 19/30 | Train Loss: 0.2203 | Train Acc: 92.67% | Test Loss: 1.4311 | Test Acc: 65.50%
Epoch 20/30 | Train Loss: 0.1872 | Train Acc: 94.33% | Test Loss: 1.4474 | Test Acc: 67.50%
Epoch 21/30 | Train Loss: 0.1723 | Train Acc: 94.00% | Test Loss: 1.5348 | Test Acc: 64.83%
Epoch 22/30 | Train Loss: 0.1681 | Train Acc: 94.12% | Test Loss: 1.6744 | Test Acc: 65.17%
Epoch 23/30 | Train Loss: 0.1434 | Train Acc: 95.08% | Test Loss: 1.7262 | Test Acc: 65.67%
Epoch 24/30 | Train Loss: 0.1285 | Train Acc: 95.96% | Test Loss: 1.9110 | Test Acc: 62.00%
Epoch 25/30 | Train Loss: 0.1133 | Train Acc: 96.58% | Test Loss: 1.7348 | Test Acc: 68.67%
Epoch 26/30 | Train Loss: 0.1607 | Train Acc: 94.46% | Test Loss: 1.7485 | Test Acc: 66.33%
Epoch 27/30 | Train Loss: 0.1224 | Train Acc: 95.79% | Test Loss: 1.8543 | Test Acc: 64.83%
Epoch 28/30 | Train Loss: 0.1632 | Train Acc: 94.21% | Test Loss: 1.9685 | Test Acc: 62.33%
Epoch 29/30 | Train Loss: 0.1312 | Train Acc: 95.50% | Test Loss: 1.8038 | Test Acc: 66.67%
Epoch 30/30 | Train Loss: 0.1229 | Train Acc: 95.58% | Test Loss: 1.6858 | Test Acc: 67.00%

In [9]:
# Load pretrained Model V3 weights from orginal saved model
# ------------------------------

load_path_v3 = "models/ewaste_cnn_v3.pth"  # change if your file is elsewhere

state_dict_v3 = torch.load(load_path_v3, map_location=device, weights_only=True)
model_v3.load_state_dict(state_dict_v3)
model_v3.to(device)
model_v3.eval()

print(f"Loaded pretrained Model V3 weights from: {load_path_v3}")


Loaded pretrained Model V3 weights from: models/ewaste_cnn_v3.pth


In [21]:
# Evaluate V3 model on test set
# ------------------------------

y_true_v3, y_pred_v3 = [], []

with torch.no_grad():
    for xb, yb in test_loader_v3:
        xb = xb.to(device)
        outputs = model_v3(xb)
        preds = outputs.argmax(1).cpu()

        y_true_v3.extend(yb.tolist())
        y_pred_v3.extend(preds.tolist())

accuracy_v3  = accuracy_score(y_true_v3, y_pred_v3)
precision_v3 = precision_score(y_true_v3, y_pred_v3, average='macro')
recall_v3    = recall_score(y_true_v3, y_pred_v3, average='macro')
f1_v3        = f1_score(y_true_v3, y_pred_v3, average='macro')

print("=== CNN V3 — Performance Metrics (Test Set) ===")
print(f"Overall Accuracy : {accuracy_v3:.3f}")
print(f"Avg Precision    : {precision_v3:.3f}")
print(f"Avg Recall       : {recall_v3:.3f}")
print(f"Avg F1 Score     : {f1_v3:.3f}")

print("\n=== Classification Report (Per Class) ===")
print(classification_report(y_true_v3, y_pred_v3, target_names=class_names_v3))



=== CNN V3 — Performance Metrics (Test Set) ===
Overall Accuracy : 0.670
Avg Precision    : 0.670
Avg Recall       : 0.670
Avg F1 Score     : 0.668

=== Classification Report (Per Class) ===
                 precision    recall  f1-score   support

        Battery       0.60      0.62      0.61        60
       Keyboard       0.77      0.77      0.77        60
      Microwave       0.65      0.55      0.59        60
         Mobile       0.58      0.63      0.61        60
          Mouse       0.52      0.45      0.48        60
            PCB       0.82      0.82      0.82        60
         Player       0.75      0.65      0.70        60
        Printer       0.58      0.58      0.58        60
     Television       0.59      0.73      0.66        60
Washing Machine       0.84      0.90      0.87        60

       accuracy                           0.67       600
      macro avg       0.67      0.67      0.67       600
   weighted avg       0.67      0.67      0.67       600



### Evaluation Interpreted (Model V3 — Custom CNN)

Model V3 reaches approximately **67% accuracy** on the Kaggle test set. The macro-averaged precision, recall, and F1 scores indicate that the model performs inconsistently across different classes. This represents a clear improvement from my earlier baseline version of the custom CNN, which originally achieved only **~43% accuracy** before architecture refinements, normalization fixes, and model tuning. While this tuned version performs noticeably better, the macro-averaged precision, recall, and F1 scores show that the model still struggles to generalize consistently across all 10 classes. This is expected for a CNN trained entirely from scratch on a relatively small image dataset, especially with visually similar e-waste categories (e.g., PCB, Player, Mobile). Overall, Model V3 is a stronger and more stable baseline than earlier attempts, but it still leaves significant room for improvement.


## 3. Model V4 — ResNet-18 Transfer Learning

In [11]:
# ==== MODEL V4: Transfer Learning with ResNet-18 ====

IMG_SIZE_V4 = 224
BATCH_SIZE_V4 = 32

train_transform_v4 = transforms.Compose([
    transforms.Resize((IMG_SIZE_V4, IMG_SIZE_V4)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet normalization
        std=[0.229, 0.224, 0.225],
    )
])

test_transform_v4 = transforms.Compose([
    transforms.Resize((IMG_SIZE_V4, IMG_SIZE_V4)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
])

train_ds_v4 = datasets.ImageFolder(root=train_dir, transform=train_transform_v4)
test_ds_v4  = datasets.ImageFolder(root=test_dir,  transform=test_transform_v4)

class_names_v4 = train_ds_v4.classes
num_classes_v4 = len(class_names_v4)
print("Classes (V4):", class_names_v4)

train_loader_v4 = DataLoader(train_ds_v4, batch_size=BATCH_SIZE_V4, shuffle=True)
test_loader_v4  = DataLoader(test_ds_v4,  batch_size=BATCH_SIZE_V4, shuffle=False)


Classes (V4): ['Battery', 'Keyboard', 'Microwave', 'Mobile', 'Mouse', 'PCB', 'Player', 'Printer', 'Television', 'Washing Machine']


### 3.1 Model V4 — ResNet-18 (Transfer Learning)

Model V4 uses **transfer learning** with a pretrained **ResNet-18**:

- Start from a ResNet-18 model pretrained on ImageNet.  
- Freeze all pretrained layers so their weights do not update.  
- Replace the final fully-connected (`fc`) layer with a new layer that outputs **10 e-waste classes**.  
- Train only this final classification layer on the e-waste training images.

This approach reuses generic visual features learned from ImageNet and adapts them to the e-waste classification task with less training data and time.


In [12]:
# 3. Pretrained ResNet-18

resnet18 = models.resnet18(pretrained=True)

# Freeze all pretrained layers
for param in resnet18.parameters():
    param.requires_grad = False

# Replace final fully-connected layer for 10 waste classes
in_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(in_features, num_classes_v4)

model_v4 = resnet18.to(device)

criterion_v4 = nn.CrossEntropyLoss()
optimizer_v4 = optim.Adam(model_v4.fc.parameters(), lr=1e-3)

print("\nV4 model ready — final layer:")
print(model_v4.fc)


C:\Users\samri\anaconda3\envs\cs171\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\samri\anaconda3\envs\cs171\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



V4 model ready — final layer:
Linear(in_features=512, out_features=10, bias=True)


In [13]:
# ---- Helper to count correct predictions ----
def compute_correct_labels_v4(outputs, labels):
    _, preds = torch.max(outputs, 1)
    return (preds == labels).sum().item(), labels.size(0)

# ---- Training loop for Model V4 (ResNet18 Transfer Learning) ----
def training_loop_v4(model, optimizer, num_epochs, train_loader, test_loader, printing=True):
    train_losses, test_losses = [], []
    train_accs,  test_accs  = [], []

    for epoch in range(num_epochs):

        # TRAIN
        model.train()
        total_train_loss = 0.0
        total_train_correct = 0
        total_train_images = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion_v4(outputs, yb)
            loss.backward()
            optimizer.step()

            correct, total = compute_correct_labels_v4(outputs, yb)
            total_train_correct += correct
            total_train_images  += total
            total_train_loss    += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_acc = total_train_correct / total_train_images
        train_losses.append(avg_train_loss)
        train_accs.append(train_acc)

        # TEST
        model.eval()
        total_test_loss = 0.0
        total_test_correct = 0
        total_test_images = 0

        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                loss = criterion_v4(outputs, yb)

                total_test_loss += loss.item()
                correct, total = compute_correct_labels_v4(outputs, yb)
                total_test_correct += correct
                total_test_images  += total

        avg_test_loss = total_test_loss / len(test_loader)
        test_acc = total_test_correct / total_test_images
        test_losses.append(avg_test_loss)
        test_accs.append(test_acc)

        if printing:
            print(
                f"Epoch {epoch+1}/{num_epochs} - "
                f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
                f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc*100:.2f}%"
            )

    return train_losses, test_losses, train_accs, test_accs


In [18]:
# OPTIONAL: How V4 was originally trained and saved
# This can stay commented out to avoid retraining.
# If you would like to run the model, uncomment the code below to run and save the model

# NUM_EPOCHS_V4 = 20
# train_losses_v4, test_losses_v4, train_accs_v4, test_accs_v4 = training_loop_v4(
#     model_v4,
#     optimizer_v4,
#     NUM_EPOCHS_V4,
#     train_loader_v4,
#     test_loader_v4,
#     printing=True
# )
#
# os.makedirs("models", exist_ok=True)
# save_path_v4 = "models/resnet18_v4.pth"
# torch.save(model_v4.state_dict(), save_path_v4)
# print(f"Model V4 saved to: {save_path_v4}")


### 3.2 Original Training Output for Model V4 (ResNet-18, 20 Epochs)

Below is the full training and test performance from the original 20-epoch run of Model V4 (ResNet-18 transfer learning). This run was completed before saving the model weights, and the logs are included here for reference.

```text
Epoch 1/20  - Train Loss: 1.5357, Train Acc: 54.79% | Test Loss: 0.9224, Test Acc: 78.83%
Epoch 2/20  - Train Loss: 0.8141, Train Acc: 79.92% | Test Loss: 0.6354, Test Acc: 84.17%
Epoch 3/20  - Train Loss: 0.6346, Train Acc: 83.46% | Test Loss: 0.5264, Test Acc: 86.00%
Epoch 4/20  - Train Loss: 0.5341, Train Acc: 85.83% | Test Loss: 0.4373, Test Acc: 88.17%
Epoch 5/20  - Train Loss: 0.4857, Train Acc: 85.83% | Test Loss: 0.3860, Test Acc: 90.33%
Epoch 6/20  - Train Loss: 0.4281, Train Acc: 87.79% | Test Loss: 0.3660, Test Acc: 90.33%
Epoch 7/20  - Train Loss: 0.3965, Train Acc: 88.79% | Test Loss: 0.3418, Test Acc: 89.67%
Epoch 8/20  - Train Loss: 0.3707, Train Acc: 89.42% | Test Loss: 0.3165, Test Acc: 90.83%
Epoch 9/20  - Train Loss: 0.3469, Train Acc: 89.75% | Test Loss: 0.3347, Test Acc: 89.00%
Epoch 10/20 - Train Loss: 0.3403, Train Acc: 90.12% | Test Loss: 0.3179, Test Acc: 90.33%
Epoch 11/20 - Train Loss: 0.3275, Train Acc: 90.00% | Test Loss: 0.3083, Test Acc: 90.33%
Epoch 12/20 - Train Loss: 0.3095, Train Acc: 91.08% | Test Loss: 0.2864, Test Acc: 91.50%
Epoch 13/20 - Train Loss: 0.2816, Train Acc: 91.79% | Test Loss: 0.2716, Test Acc: 91.83%
Epoch 14/20 - Train Loss: 0.3058, Train Acc: 90.79% | Test Loss: 0.2776, Test Acc: 91.17%
Epoch 15/20 - Train Loss: 0.2883, Train Acc: 91.17% | Test Loss: 0.2572, Test Acc: 92.33%
Epoch 16/20 - Train Loss: 0.2958, Train Acc: 90.33% | Test Loss: 0.2549, Test Acc: 92.83%
Epoch 17/20 - Train Loss: 0.2828, Train Acc: 91.79% | Test Loss: 0.2699, Test Acc: 91.83%
Epoch 18/20 - Train Loss: 0.2661, Train Acc: 92.12% | Test Loss: 0.2457, Test Acc: 92.67%
Epoch 19/20 - Train Loss: 0.2604, Train Acc: 92.46% | Test Loss: 0.2494, Test Acc: 92.00%
Epoch 20/20 - Train Loss: 0.2691, Train Acc: 91.88% | Test Loss: 0.2602, Test Acc: 91.83%



In [16]:
# Load pretrained Model V4 weights
# ------------------------------

load_path_v4 = "models/resnet18_v4.pth" 

state_dict_v4 = torch.load(load_path_v4, map_location=device, weights_only=False)
model_v4.load_state_dict(state_dict_v4)
model_v4.to(device)
model_v4.eval()

print(f"Loaded pretrained Model V4 weights from: {load_path_v4}")


Loaded pretrained Model V4 weights from: models/resnet18_v4.pth


In [17]:
# Evaluate V4 model on test set
# ------------------------------

y_true_v4, y_pred_v4 = [], []

with torch.no_grad():
    for xb, yb in test_loader_v4:
        xb = xb.to(device)
        outputs = model_v4(xb)
        preds = outputs.argmax(1).cpu()

        y_true_v4.extend(yb.tolist())
        y_pred_v4.extend(preds.tolist())

accuracy_v4  = accuracy_score(y_true_v4, y_pred_v4)
precision_v4 = precision_score(y_true_v4, y_pred_v4, average='macro')
recall_v4    = recall_score(y_true_v4, y_pred_v4, average='macro')
f1_v4        = f1_score(y_true_v4, y_pred_v4, average='macro')

print("=== ResNet-18 V4 — Performance Metrics (Test Set) ===")
print(f"Overall Accuracy : {accuracy_v4:.3f}")
print(f"Avg Precision    : {precision_v4:.3f}")
print(f"Avg Recall       : {recall_v4:.3f}")
print(f"Avg F1 Score     : {f1_v4:.3f}")

print("\n=== Classification Report (Per Class) ===")
print(classification_report(y_true_v4, y_pred_v4, target_names=class_names_v4))


=== ResNet-18 V4 — Performance Metrics (Test Set) ===
Overall Accuracy : 0.918
Avg Precision    : 0.922
Avg Recall       : 0.918
Avg F1 Score     : 0.918

=== Classification Report (Per Class) ===
                 precision    recall  f1-score   support

        Battery       0.89      0.90      0.89        60
       Keyboard       0.97      0.98      0.98        60
      Microwave       0.85      0.97      0.91        60
         Mobile       0.97      0.93      0.95        60
          Mouse       0.95      0.98      0.97        60
            PCB       0.91      0.98      0.94        60
         Player       0.82      0.93      0.88        60
        Printer       0.96      0.80      0.87        60
     Television       0.94      0.78      0.85        60
Washing Machine       0.96      0.92      0.94        60

       accuracy                           0.92       600
      macro avg       0.92      0.92      0.92       600
   weighted avg       0.92      0.92      0.92       600



### Evaluation Interpreted (Model V4 — ResNet-18 Transfer Learning)

Model V4 significantly outperforms the custom CNN, achieving **over 90% test accuracy** with strong macro-averaged precision, recall, and F1. The per-class report shows much more balanced performance across categories. This improvement is expected because ResNet-18 starts with pretrained ImageNet features—allowing it to recognize edges, textures, and object shapes far more effectively than a model trained from scratch. Fine-tuning only the final layer enables the model to adapt these learned features to the e-waste dataset, resulting in better generalization and reliability compared to V3.


## 4. Final Summary

In this notebook, I constructed, trained, and evaluated two deep learning models for the e-waste image classification task:

- **Model V3 — Custom CNN (trained from scratch at 128×128 resolution)**
- **Model V4 — ResNet-18 Transfer Learning (pretrained on ImageNet and fine-tuned on the final layer)**

All dataset preparation steps were performed in the previous notebook, `01_data_preparation_ewaste.ipynb`, which organizes the training and test splits used here.

**Model V3 (Custom CNN).**  
After tuning the architecture, normalization, and optimizer settings, the improved CNN achieved **~67% accuracy**, which is a notable increase from the earlier baseline version that originally performed around **43% accuracy**. Although the model demonstrates learning, the class-wise metrics show that it still struggles with generalization, especially for visually similar categories, which is expected  when training from scratch on a modest dataset.

**Model V4 (ResNet-18 Transfer Learning).**  
The transfer-learning model provided a substantial performance boost. By leveraging pretrained ImageNet features and training only the final classification layer, ResNet-18 achieved **over 90% test accuracy**, with strong macro-averaged precision, recall, and F1 scores. This result highlights the effectiveness of pretrained CNNs when labeled data is limited.

Overall, this notebook shows the progression from a basic CNN baseline to a significantly stronger transfer-learning model, illustrating how model capacity and prior knowledge influence performance on real-world e-waste classification.

A deeper investigation—including training/validation curves, confusion matrices, per-class visualizations, and a bonus evaluation on the separate hand-curated validation set—will be presented in the next notebook:  
**`03_analysis_visualization_ewaste.ipynb`**.

